# Position Encoding: Letting the Model Sense Word Order

> In the previous section, we used Embedding to turn token IDs into vectors — each token ID looks up a fixed high-dimensional vector from a table. But there is a problem: the same token produces the exact same vector regardless of where it appears in the sentence. In other words, the model at this stage cannot distinguish between "the cat" and "cat the".
>
> This section introduces Position Encoding, which prepares a unique vector for each position and adds it to the Token Embedding. Starting from concrete examples, we first examine what goes wrong when the model lacks position information, then step through the sinusoidal position encoding scheme, and finally assemble the complete Transformer input layer.

"mao zhui gou" (the cat chases the dog) and "gou zhui mao" (the dog chases the cat) contain the exact same three characters, but because the order is different, the meanings are completely opposite. "Ni xi huan wo" (you like me) and "wo xi huan ni" (I like you) simply swap two words, yet one means the other person likes me and the other means I like the other person. The same holds in English: "the cat sat" describes a cat sitting down, while "sat the cat" does not look like a normal sentence.

These examples point to a common fact: the same token can play different roles at different positions. Without knowing positions, the model cannot understand such differences.


## 1. Why Position Information Matters

The Embedding layer we built in the previous section actually contains no position information at all: the lookup only consults the token ID, not the position. No matter where a given ID appears, the same row is pulled from the matrix. The code below verifies this.


In [ ]:
import torch
import torch.nn as nn

# Use a tiny Embedding to verify: does the same token get the same vector at different positions?
torch.manual_seed(42)

vocab_size = 10    # 10 tokens
d_model = 4       # 4-dimensional vectors, easy to print and inspect

emb_table = nn.Embedding(vocab_size, d_model)

# Construct an input: token 0 appears at position 0 and position 3
# with a few different tokens in between
input_ids = torch.tensor([[0, 2, 5, 0, 7]])  # [batch=1, seq_len=5]

vectors = emb_table(input_ids)  # [1, 5, 4]

print("=== Input token IDs ===")
print(input_ids)
print(f"\n=== Embedding lookup results (shape: {vectors.shape}) ===")
for pos in range(5):
    tid = input_ids[0, pos].item()
    vec = vectors[0, pos].detach()
    print(f"Position {pos}, token {tid}: {vec.tolist()}")

print(f"\n=== Key observation ===")
v0 = vectors[0, 0].detach()
v3 = vectors[0, 3].detach()
print(f"Token 0 at position 0: {v0.tolist()}")
print(f"Token 0 at position 3: {v3.tolist()}")
print(f"Are they identical? {torch.equal(v0, v3)}")
print(f"\nToken 0 receives the exact same vector at position 0 and position 3.")
print(f"The model cannot distinguish their positions at this stage.")


The output confirms the problem: when only a token embedding lookup is performed, the same token yields identical vectors at position 0 and position 3. To the model, the two occurrences of "the" are indistinguishable.

The severity of this problem depends on how important word order is in a given language. In Chinese, "wo xi huan ni" (I like you) and "ni xi huan wo" (you like me) contain the exact same tokens, but the meaning hinges on who comes first. In English, "dog bites man" is routine news, while "man bites dog" is front-page material. In Japanese, word order differs from both Chinese and English, yet grammatical relationships between words are still marked by position. No matter the language, position information is not an optional decoration — it is foundational to understanding sentence structure.

Therefore, before feeding the token ID sequence into the Transformer, we need to attach additional position information. The next question is: how should we represent positions?


## 2. Sinusoidal Position Encoding

### Starting with the Most Straightforward Approach

The most direct idea for identifying each position is to simply use the position number itself. Position 0 gets the vector `[0, 0, 0, ...]`, position 1 gets `[1, 1, 1, ...]`, and position 100 gets `[100, 100, 100, ...]`. Let us implement this scheme first and see what problems arise.


In [ ]:
import torch

# Approach 1: use position numbers directly as position vectors
def get_naive_encoding(seq_len, d_model):
    """Naive position encoding: fill every dimension with the position number."""
    pe = torch.zeros(seq_len, d_model)
    for pos in range(seq_len):
        pe[pos, :] = pos  # All dimensions get the same value: pos
    return pe

naive_pe = get_naive_encoding(seq_len=10, d_model=4)
print("=== Approach 1: Direct position numbers ===")
print(naive_pe)
print()
print("Problem 1: Values grow with position")
print(f"  Position 0 values: {naive_pe[0].tolist()}, norm = {naive_pe[0].norm():.2f}")
print(f"  Position 5 values: {naive_pe[5].tolist()}, norm = {naive_pe[5].norm():.2f}")
print(f"  Position 9 values: {naive_pe[9].tolist()}, norm = {naive_pe[9].norm():.2f}")
print(f"  -> Norm grows from 0 to {naive_pe[9].norm():.2f}, a huge difference")
print(f"  -> Neural networks work better with bounded values, not unbounded growth")
print()
print("Problem 2: All dimensions carry the same information")
print(f"  Each position has the same value across all 4 dimensions, effectively only 1 information source")
print(f"  -> Wastes the d_model-dimensional space")

Using position numbers directly has two clear flaws: the values grow without bound as positions increase, and all dimensions carry identical information.

One improvement is to learn a vector for each position (learnable position embedding), which is the approach used by the GPT series. Its limitation is that it can only represent positions up to the maximum length seen during training — if the model was trained on sequences of at most 512 tokens, it has no reference for position 513.

The original Transformer paper chose a different approach: using sin and cos functions to hand-craft position vectors. There are three reasons for choosing trigonometric functions:

1. **Bounded values**: sin and cos values always stay in [-1, 1], never diverging as positions grow.
2. **Extrapolation**: Because they are continuous functions, positions not seen during training can be computed directly.
3. **Implicit relative position relationships**: Using trigonometric identities, the encoding for position pos+k can be obtained from the encoding of position pos via a linear transformation. This means the model can potentially learn "how far apart two tokens are" rather than just memorizing absolute position numbers.

The formula is:

$$
PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d}}\right), \quad
PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d}}\right)
$$

Here pos is the position number, i is the dimension index, and d is the total number of dimensions. Even-numbered dimensions use sin, and odd-numbered dimensions use cos.

There is no need to memorize the formula just yet. Let us first work through a set of concrete numbers by hand to see what this encoding actually looks like.


In [ ]:
import math
import torch

# Hand calculation: d_model=4, position encodings for positions 0-3
# Formula: PE(pos, 2i) = sin(pos / 10000^(2i/d))
#          PE(pos, 2i+1) = cos(pos / 10000^(2i/d))

d = 4  # Total dimensions

print("=== Parameter computation ===")
print(f"d_model = {d}")
print()

# First compute the denominator 10000^(2i/d) for each dimension pair
for i in range(d // 2):
    freq = 10000 ** (2 * i / d)
    print(f"Dimension pair {i}: 10000^({2*i}/{d}) = 10000^{2*i/d:.1f} = {freq:.4f}")

print()
print("=== Per-position computation ===")
for pos in range(4):
    print(f"\nPosition {pos}:")
    for i in range(d // 2):
        freq = 10000 ** (2 * i / d)
        val_sin = math.sin(pos / freq)
        val_cos = math.cos(pos / freq)
        print(f"  Dimension {2*i} (sin): sin({pos} / {freq:.4f}) = sin({pos/freq:.4f}) = {val_sin:.4f}")
        print(f"  Dimension {2*i+1} (cos): cos({pos} / {freq:.4f}) = cos({pos/freq:.4f}) = {val_cos:.4f}")

print()
print("=== Summary table ===")
# Use the function to cross-check
pe_small = torch.zeros(4, d)
position = torch.arange(4).unsqueeze(1).float()
div_term = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
pe_small[:, 0::2] = torch.sin(position * div_term)
pe_small[:, 1::2] = torch.cos(position * div_term)
header = f"{'Pos':>4} | {'dim 0 (sin)':>12} | {'dim 1 (cos)':>12} | {'dim 2 (sin)':>12} | {'dim 3 (cos)':>12}"
print(header)
print("-" * len(header))
for pos in range(4):
    vals = " | ".join(f"{pe_small[pos, j]:>12.6f}" for j in range(d))
    print(f"{pos:>4} | {vals}")
print()
print("Key observations:")
print(f"  Dimensions 0-1 denominator = 1.0 -> wavelength = 2pi ~ 6.28, changes fast, distinguishes adjacent positions")
print(f"  Dimensions 2-3 denominator = 100.0 -> wavelength = 200pi ~ 628, changes slowly, conveys long-range relationships")
print(f"  Different dimensions change at different speeds, combining to give each position a unique \"fingerprint\"")

The hand calculation clearly shows how sinusoidal position encoding works. Let us summarize the observations:

Each dimension corresponds to a sine (or cosine) wave. The smaller the dimension index, the smaller the denominator, and the higher the wave's frequency — the value changes noticeably with even a small shift in position, which is useful for distinguishing adjacent positions. The larger the dimension index, the larger the denominator, and the lower the frequency — the value changes slowly, which is useful for conveying long-range positional relationships.

An intuitive analogy: low-frequency dimensions are like the hour hand of a clock, changing slowly and telling us roughly which time period we are in; high-frequency dimensions are like the second hand, changing rapidly and giving us precise timing. Combining both types of information uniquely determines a moment in time. Sinusoidal position encoding works on the same principle: waves of different frequencies combine to generate a unique vector for each position.

The choice of 10000 as the base was an empirical decision in the original paper. A larger base creates a bigger frequency gap between high and low frequencies, allowing the encoding to cover a wider range of positions. The value 10000 performs well for common sequence lengths (from tens to thousands of tokens).

Now let us implement the complete encoding function and examine the overall structure of the encoding with larger parameters.


In [ ]:
import math
import torch

def get_sinusoidal_encoding(seq_len, d_model):
    """
    Sinusoidal position encoding: each position gets a unique vector
    generated by sin/cos waves of different frequencies.
    
    PE(pos, 2i)   = sin(pos / 10000^(2i/d))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d))
    """
    position = torch.arange(seq_len).unsqueeze(1)  # [seq_len, 1]
    div_term = torch.exp(
        torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
    )
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)   # Even dimensions: sin
    pe[:, 1::2] = torch.cos(position * div_term)   # Odd dimensions: cos
    return pe

# Generate encodings for 10 positions, 8 dimensions
pe = get_sinusoidal_encoding(seq_len=10, d_model=8)
print(f"Position encoding shape: {pe.shape}  <- 10 positions x 8 dimensions each")
print(f"First 3 positions:\n{pe[:3]}")

**Visualization: What does position encoding look like?**

The numbers printed above do not make the pattern easy to see. Below we plot the encoding for 50 positions and 32 dimensions. The left panel is a heatmap: each row is a position, each column is a dimension, and the color represents the value. The right panel selects several dimensions and plots how their values change across positions — we can see different dimensions oscillating at different frequencies.


In [ ]:
import matplotlib.pyplot as plt

# Visualize position encoding: heatmap + waveform plot
pe_viz = get_sinusoidal_encoding(seq_len=50, d_model=32)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: heatmap — x-axis=dimension, y-axis=position, color=value
im = axes[0].imshow(pe_viz.numpy(), aspect='auto', cmap='RdBu')
axes[0].set_xlabel('Embedding dim (0-31)'); axes[0].set_ylabel('Position (0-49)')
axes[0].set_title('Positional encoding heatmap\n(blue=-1, red=+1)')
plt.colorbar(im, ax=axes[0])

# Right: select a few dimensions and see how their values change across positions
# Low dimensions: short wavelength (fast change -> distinguishes neighbors)
# High dimensions: long wavelength (slow change -> distinguishes distant positions)
for dim_idx in [0, 1, 4, 8, 16]:
    axes[1].plot(range(50), pe_viz[:, dim_idx].numpy(), label=f'Dim {dim_idx}')
axes[1].set_xlabel('Position'); axes[1].set_ylabel('Encoding value')
axes[1].set_title('Encoding waves across positions')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**How to read the heatmap**

In the left panel, the x-axis is the dimension index (0-31), the y-axis is the position index (0-49), and the color represents the encoding value: red approaches +1, blue approaches -1.

The most striking feature is the dense left half, the sparse right half, and an arc in the middle. Recall the formula $PE(pos, 2i) = \sin(pos / 10000^{2i/d})$: the larger the dimension index $i$, the larger the denominator, and the longer the wavelength. Within 50 positions, different dimensions behave quite differently:

- **dim 0-7 (left half)**: Short wavelength (approximately 6-15). Fifty positions accommodate 3-8 complete sin/cos cycles, creating dense alternating stripes of red and blue.
- **dim 8-15 (middle)**: Wavelength grows longer (approximately 60-150). Fifty positions are not enough to complete a full cycle. The sin/cos only traverses a portion of its rising or falling phase, appearing as an arc in the heatmap — essentially an incomplete half-wave.
- **dim 16-31 (right half)**: Extremely long wavelength (600+). Within 50 positions the value barely changes, appearing as a uniform color block in the heatmap.

The "outer arc" can be understood as follows: the denominator $10000^{2i/d}$ grows **exponentially** with $i$, so the frequency decrease from high to low is not uniform. From dim 0 to dim 8, the frequency drops by about 10x (from ~1 to ~0.1); from dim 8 to dim 16, it drops another 10x. In the dim 8-14 range, each dimension's wave has only traversed part of its rising or falling arc. These incomplete half-waves stitched together form the outward-bulging curve.

The curves in the right panel can be used for cross-verification. Dimension 0 (blue line) oscillates about 8 times across 50 positions; dimension 8 (red line) covers less than half of a rising phase; dimension 16 (purple line) is nearly a straight line. Imagine standing these curves upright and placing them side by side — that would reproduce the full picture of the left heatmap.

**Why do we need so many different frequencies?** The gradient from dense to sparse in the heatmap is not accidental — it follows the same principle as binary counting. Using binary to represent 0-15:

```
 0: 0 0 0 0     4: 0 1 0 0     8: 1 0 0 0    12: 1 1 0 0
 1: 0 0 0 1     5: 0 1 0 1     9: 1 0 0 1    13: 1 1 0 1
 2: 0 0 1 0     6: 0 1 1 0    10: 1 0 1 0    14: 1 1 1 0
 3: 0 0 1 1     7: 0 1 1 1    11: 1 0 1 1    15: 1 1 1 1
```

Look first at the leftmost group (0-3). Each row has four digits; if we look only at the rightmost column: `0, 1, 0, 1` — it flips between 0 and 1 at every step. Now the second column from the right: `0, 0, 1, 1` — it flips only every two steps. Move one column further left: `0, 0, 0, 0` — within four steps there is not even a single flip, and we need to look at more digits to observe any change.

Extending the view to all 16 numbers, the complete flip pattern of each column becomes clear:

- **bit 0** (rightmost column): `0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1` — flips every step, period of 2.
- **bit 1** (second column from the right): `0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1` — flips every two steps, period of 4.
- **bit 2** (third column from the right): `0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1` — flips every four steps, period of 8.
- **bit 3** (leftmost column): `0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1` — flips every eight steps, period of 16.

Each move one bit to the left halves the flip speed: the period doubles from 2 to 4, 8, 16. Any single bit can only take the values 0 or 1, which carries limited information. But four bits flipping at different speeds combine to give every number from 0 to 15 a unique encoding. As a check: reading the table top to bottom gives the binary representations of 0 through 15, and no two rows are identical.

Sinusoidal position encoding does exactly the same thing, except it replaces the 0/1 discrete flips with continuous sin/cos oscillations. The low dimensions on the left of the heatmap correspond to the low bits of binary — they change quickly, producing a different value at every step. The high dimensions on the right correspond to high bits — they change slowly, requiring a long distance before the value shifts noticeably. This design has two direct benefits.

First, each position receives a unique vector (just as every integer has a unique binary representation).

Second, the trigonometric identity $\sin(a+b) = \sin a \cos b + \cos a \sin b$ (and the cosine equivalent) brings a special property: the encoding for position pos+k can be obtained from the encoding of position pos via a single linear transformation, and this transformation depends only on the distance k, not on the absolute position pos. What does this mean? Suppose the model sees the position encoding vectors of two tokens. If it can recognize the transformation between those two vectors, it can infer how far apart they are — regardless of where they appear in the sentence. In other words, the model does not have to memorize "this word is at position 5, that word is at position 7"; it has the opportunity to learn directly "these two words are 2 positions apart".

> The binary analogy comes from Kazemnejad's blog post [Transformer Architecture: The Positional Encoding](https://kazemnejad.com/transformer_architecture_positional_encoding/), which provides an excellent intuitive explanation of sinusoidal position encoding.


In [ ]:
# Print the flip pattern of binary digit by digit, to feel "different change speeds -> unique representation"
print("=== Flip patterns of the four columns in binary 0-15 ===\n")
print(f"{'Dec':>4}  {'bit3':>4}  {'bit2':>4}  {'bit1':>4}  {'bit0':>4}")
print("-" * 32)
for n in range(16):
    bits = f"{n:04b}"
    print(f"{n:>4}  {bits[0]:>4}  {bits[1]:>4}  {bits[2]:>4}  {bits[3]:>4}")

print()
print("=== Flip period of each column ===")
for bit in range(4):
    pattern = "".join(str((n >> bit) & 1) for n in range(16))
    period = 2 ** (bit + 1)
    half = period // 2
    print(f"  bit {bit}: {pattern}  -> flips every {half} steps, period = {period}")

print()
print("The periods of the four columns are 2, 4, 8, 16, doubling each time.")
print("The wavelengths of sinusoidal encoding grow exponentially by the same rule, just replacing 0/1 with continuous values.")

### Why Addition Instead of Concatenation

Now that we have position information, the next decision is how to combine it with the Token Embedding. The two most intuitive approaches are addition and concatenation.

Concatenation appends the position vector to the end of the token vector, producing a longer vector. For example, if the token vector is 4-dimensional and the position vector is also 4-dimensional, concatenation yields an 8-dimensional vector. The information indeed stays separate, but the doubled dimensionality means that every subsequent layer's parameters and computation cost also double.

Addition is more concise: the two vectors are simply added element-wise, keeping the same dimensionality. The prerequisite is that the subsequent linear layers (and Attention layers) can extract both token information and position information from the summed vector — which is possible because a linear transformation can learn to separate two additive signals.

The standard Transformer uses addition, primarily for efficiency: the dimensionality does not increase, so computation cost does not balloon. In practice, with sufficient training, addition performs no worse than concatenation.

## 3. Assembly: Token Embedding + Position Encoding

Now let us combine Token Embedding and Position Encoding into a single PyTorch Module.

Input: token IDs `[batch_size, seq_len]`

Output: vectors `[batch_size, seq_len, embed_dim]`

Each position in each sample gets a vector that contains both the token's semantic information and its position information.


In [ ]:
import torch
import torch.nn as nn

class TokenEmbedding(nn.Module):
    """Token ID -> vector: Embedding lookup + position encoding"""
    
    def __init__(self, vocab_size, d_model, max_seq_len=512):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        # Position Encoding is precomputed, not trained (register_buffer)
        pe = get_sinusoidal_encoding(max_seq_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x: [batch, seq_len] -> output: [batch, seq_len, d_model]
        seq_len = x.shape[1]
        token_vecs = self.token_emb(x)         # Lookup
        pos_vecs = self.pe[:seq_len, :]        # Slice position encodings
        return token_vecs + pos_vecs            # Addition (not concatenation)

# Test: use fixed input to ensure the same token at different positions
test_vocab_size, test_d_model = 20, 8
emb_module = TokenEmbedding(test_vocab_size, test_d_model)

# token 0 appears at both position 0 and position 3
test_input = torch.tensor([
    [0, 1, 2, 0, 3],   # Same token(0) at position 0 and position 3
    [5, 4, 0, 6, 0],   # Same token(0) at position 2 and position 4
])
output = emb_module(test_input)

print(f"Input:\n{test_input}")
print(f"Output shape: {output.shape}  -> [batch=2, seq_len=5, d_model={test_d_model}]")

# Verify: in batch 0, positions 0 and 3 are both token 0
tid_0 = test_input[0, 0].item()
tid_3 = test_input[0, 3].item()
print(f"\nPosition 0 token ID = {tid_0}, position 3 token ID = {tid_3}")
print(f"Same token, but with different position encodings added:")
print(f"  Position 0: {output[0, 0, :3].tolist()}")
print(f"  Position 3: {output[0, 3, :3].tolist()}")
print(f"  -> Different! Position encoding gives the same token different representations at different positions.")

In [ ]:
import torch
import matplotlib.pyplot as plt

# Visual comparison: before and after adding position encoding,
# the same token at different positions has different vectors
torch.manual_seed(42)
viz_vocab, viz_dim = 20, 8
viz_module = TokenEmbedding(viz_vocab, viz_dim)

# token 0 appears at position 0 and position 3
viz_input = torch.tensor([[0, 1, 2, 0, 3]])

# Token embedding only (without position encoding)
token_only = viz_module.token_emb(viz_input)

# With position encoding added
with torch.no_grad():
    token_plus_pos = viz_module(viz_input)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Compare token 0 at position 0 and position 3
positions = [0, 3]  # Two positions where token 0 appears
colors = ['#4C72B0', '#DD8452']
labels = ['pos 0 (token 0)', 'pos 3 (token 0)']

width = 0.3
# Left: token embedding only
for idx, pos in enumerate(positions):
    vals = token_only[0, pos].detach().numpy()
    axes[0].bar([j + idx * width for j in range(viz_dim)], vals,
                width=width, color=colors[idx], label=labels[idx], alpha=0.8)
axes[0].set_title('Token Embedding only\n(same token = same bars)')
axes[0].set_xlabel('Dimension')
axes[0].set_ylabel('Value')
axes[0].legend()

# Right: token embedding + position encoding
for idx, pos in enumerate(positions):
    vals = token_plus_pos[0, pos].detach().numpy()
    axes[1].bar([j + idx * width for j in range(viz_dim)], vals,
                width=width, color=colors[idx], label=labels[idx], alpha=0.8)
axes[1].set_title('Token Embedding + Position Encoding\n(same token, different position = different bars)')
axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('Value')
axes[1].legend()

plt.tight_layout()
plt.show()

# Numerical confirmation
print("=== Numerical confirmation ===")
print(f"Token 0 at position 0 (raw Embedding): {token_only[0, 0, :4].tolist()}")
print(f"Token 0 at position 3 (raw Embedding): {token_only[0, 3, :4].tolist()}")
print(f"Identical with raw Embedding? {torch.allclose(token_only[0, 0], token_only[0, 3])}")
print()
print(f"Token 0 at position 0 (+ position encoding): {token_plus_pos[0, 0, :4].tolist()}")
print(f"Token 0 at position 3 (+ position encoding): {token_plus_pos[0, 3, :4].tolist()}")
print(f"Identical after adding position encoding? {torch.allclose(token_plus_pos[0, 0], token_plus_pos[0, 3])}")
print()
print("Position encoding solves the problem raised at the beginning: the same token now has different representations at different positions.")

## Appendix: The Embedding Scaling Convention

The original Transformer paper also multiplies by $\sqrt{d\_model}$ after the lookup:

```python
embedding_output = self.token_emb(x) * math.sqrt(d_model)
```

Why is this step needed? Think of the addition of Embedding and position encoding as two people speaking at the same time: Embedding says "what this word means" and position encoding says "which position this word is at". If both speak at comparable volumes, the listener has trouble distinguishing the primary from the secondary.

When $d\_model = 512$, this is exactly the situation. `nn.Embedding` initializes with a normal distribution of roughly unit standard deviation, sampling each dimension independently, so the vector norm is approximately $\sqrt{512} \approx 22.6$. Position encoding values oscillate between $[-1, 1]$, with a norm of about $\sqrt{512 \times 0.5} \approx 16$. The gap is less than 2x — position information is at the same order of magnitude as semantic information, so adding them directly puts the two signals on equal footing.

Multiplying by $\sqrt{512} \approx 22.6$ amplifies the Embedding norm to about 512, while position encoding remains at 16. Now the semantic signal is about 30x stronger than the position signal — "meaning" holds the microphone, while "position" recedes to the background and only provides fine-tuning. This is the same reasoning as dividing by $\sqrt{d_k}$ in Attention: controlling magnitudes so that subsequent softmax and linear layers operate in numerically stable ranges.

GPT-2 and GPT-3 use learnable position embeddings and do not rely on this scaling. Later models (such as LLaMA) switched to RoPE rotary position encoding, which likewise does not need this scaling. Understanding this convention is helpful for reading the original Transformer implementation.


## Appendix: The Batch Dimension

In the code above, the input shape went from `[seq_len]` to `[batch_size, seq_len]`. This is because during training, samples are not processed one at a time but in batches.

```
batch_size = 2

Sample 0: [the, cat, sat]
Sample 1: [dog, log, mat]

Combined into a batch:
[[0, 1, 2],
 [5, 6, 4]]
```

GPUs excel at processing many samples simultaneously, and batches exist to take full advantage of this capability.


In [ ]:
import torch

# Intuitive feel for batches: process multiple samples at once
batch = torch.tensor([
    [0, 1, 2, 3, 0],  # the cat sat on the
    [5, 4, 6, 1, 2],  # dog mat log cat sat
])

emb_output = emb_module(batch)
print(f"Input shape: {batch.shape} = [batch_size=2, seq_len=5]")
print(f"Output shape: {emb_output.shape} = [2, 5, d_model={test_d_model}]")
print(f"-> GPU processes 2 samples at once, faster than one at a time")

## Summary

What we learned in this section:

- The same token at different positions produces identical Embeddings — the model needs additional position information to distinguish order
- Using position numbers directly has two problems: values grow without bound, and all dimensions carry redundant information
- Sinusoidal position encoding generates unique vectors for each position using sin/cos waves of different frequencies
- sin/cos values always stay in [-1, 1], never diverging as positions grow
- Positions not seen during training can still be computed directly; this is called "computability to longer positions", which does not mean the model necessarily performs well on longer contexts
- Waves of different frequencies combine: low frequencies convey long-range relationships, high frequencies distinguish adjacent positions
- Final input = Token Embedding + Position Encoding (addition, dimensionality unchanged)
- The batch dimension lets the GPU process multiple samples at once, improving training efficiency

The next section moves into Self-Attention: given vectors that carry both semantic and position information, how does the model let tokens "see" each other?


## Exercises

> You can use AI to ask for hints, break down steps, or check your direction, but it is not recommended to have AI "solve the exercise for you" directly.


### Exercise 1: Adding Position Encoding

When the same token appears at different positions, the token embeddings are identical; they only become different after adding position encoding.

Hint: The final input is typically `token_vectors + position_vectors`.


In [ ]:
import torch

# Exercise 1: add Token Embedding and Position Encoding
token_vectors = torch.tensor([
    [1.0, 1.0],
    [1.0, 1.0],
])
position_vectors = torch.tensor([
    [0.0, 0.1],
    [0.2, 0.3],
])

# Exercise 1: add Token Embedding and Position Encoding
final_vectors = token_vectors + position_vectors

assert not isinstance(final_vectors, str), 'Replace the placeholder before running the assertion.'
expected = torch.tensor([[1.0, 1.1], [1.2, 1.3]])
assert torch.allclose(final_vectors, expected), final_vectors
assert not torch.allclose(final_vectors[0], final_vectors[1])
print('Exercise 1 passed: you remember why position encoding is necessary')

### Exercise 2: Embedding Mask After Padding

During training, sentences of different lengths are often padded to the same length. The vector at the padding position should typically not influence learning.

Hint: Here we manually zero out the vectors at PAD positions to simulate "ignoring PAD".


In [ ]:
import torch

# Exercise 2: zero the vectors at PAD positions
embeddings = torch.tensor([
    [[1.0, 1.0], [2.0, 2.0], [9.0, 9.0], [9.0, 9.0]],
])
attention_mask = torch.tensor([[1, 1, 0, 0]])

# Exercise 2: zero the vectors at PAD positions
masked_embeddings = embeddings * attention_mask.unsqueeze(-1)

assert not isinstance(masked_embeddings, str), 'Replace the placeholder before running the assertion.'
expected = torch.tensor([[[1.0, 1.0], [2.0, 2.0], [0.0, 0.0], [0.0, 0.0]]])
assert torch.equal(masked_embeddings, expected), masked_embeddings
print('Exercise 2 passed: you understand padding in batched training')

**Exercise 3: Periodicity of Sinusoidal Position Encoding** The formula for sinusoidal position encoding is $PE(pos, 2i) = \sin(pos / 10000^{2i/d})$. Suppose $d = 4$ (2 dimension pairs). Compute the encoding value of the first dimension pair ($i=0$) for positions $pos = 0$ and $pos = 10000$. Observe: what is the period of the first dimension pair? (i.e., the change in $pos$ required for the $\sin$ function to complete one full cycle.)

Hint: When $i=0$, $\theta = pos / 10000^{0} = pos$, so the period of $\sin(pos)$ is $2\pi \approx 6.28$.


In [ ]:
# Exercise 3: Periodicity of sinusoidal position encoding
import math

d = 4
i = 0
pe_0 = None       # TODO: compute the sine value at position 0
pe_10000 = None   # TODO: compute the sine value at position 10000
period = None     # TODO: period of the first dimension pair, 2*pi

assert pe_0 is not None
assert pe_10000 is not None
assert period is not None
assert abs(pe_0 - 0.0) < 0.001
assert abs(pe_10000 - math.sin(10000)) < 0.001
assert abs(period - 2 * math.pi) < 0.01
print(f"pe(0, i=0) = sin(0) = {pe_0:.4f}")
print(f"pe(10000, i=0) = sin(10000) = {pe_10000:.4f}")
print(f"Period = {period:.2f} (about {period:.1f} positions)")
print("Low dimensions have short periods; high dimensions have long periods.")
print("Exercise 3 passed")


## References

- Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762), 2017 — The original Transformer paper; sinusoidal position encoding and the Embedding scaling convention both come from this paper
- Harvard NLP, [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/) — A line-by-line implementation of the original paper
- Kazemnejad, [Transformer Architecture: The Positional Encoding](https://kazemnejad.com/transformer_architecture_positional_encoding/) — An intuitive explanation of sinusoidal position encoding, including the binary analogy


## Appendix: Extrapolation of Sinusoidal Position Encoding

We noted earlier that sinusoidal position encoding can "extrapolate" — trained on 512 positions, it can still compute a valid encoding for position 513 at inference time. This is not something to take for granted. To understand why, we need to compare the underlying mechanisms of the two approaches.

**The limitation of learnable position encoding.** The learnable position embedding used by GPT-2 and BERT is essentially a lookup table: at training time max_seq_len is set to 512, so the model learns 512 vectors indexed 0 to 511. When position 513 arrives, that row simply does not exist in the table — there is no learned vector to look up, and no sensible default value to fill in. The maximum length the model can handle is fixed at training time. GPT-3 uses the same scheme; its context window length (2048) was determined during training and cannot be exceeded at inference.

**The advantage of sinusoidal position encoding.** Its encoding is computed directly by a formula: PE(pos) = sin(pos / 10000^(2i/d)). Whether pos is 100 or 100000, plugging it into the formula yields a valid vector. The difference between a lookup table and a formula is like the difference between a 512-page dictionary and a set of word-formation rules: words missing from the dictionary are simply unavailable, but the formation rules can produce a result for any word.

From another angle, sinusoidal encoding has an additional mathematical property: there is a deterministic linear relationship between the encoding of position pos and the encoding of position pos+k (derived from the trigonometric identity $\sin(a+b) = \sin a \cos b + \cos a \sin b$). This means the "pattern of encoding differences between adjacent positions" that the model learns during training still holds beyond the training length. Learnable encodings have no such structure — the relationship between the vectors at positions 511 and 512 depends entirely on the parameters learned during training, with no prior rule to follow.

"The encoding can be computed" does not mean "the model performs equally well on longer sequences". During training the model has only seen encoding patterns within the 512-position range; beyond that, the statistical properties of attention weights, layer normalization, and other modules may shift. Sinusoidal encoding provides mathematical computability, but the actual extrapolation effect still depends on the overall design of the model.

For this reason, subsequent research has proposed a variety of improvements. ALiBi directly adds a bias that grows linearly with distance to the Attention scores; the formula itself has no length limit, giving it stronger extrapolation than sinusoidal encoding. RoPE (Rotary Position Embedding) encodes relative positions using a rotation matrix and is adopted by mainstream models such as LLaMA and Mistral; combined with position interpolation techniques, it can extend the context window to multiples of the training length. One of the original design motivations for these schemes is precisely to keep position encoding effective on longer sequences.
